main changes:
- updated some variable names for readability across functions
- fixed comparison typo in the PLE function to skip the satisfied clauses
- added early conflict detection in the UP logic, and loop restart when a unit clause is found
- some variable caching, handling empty symbols and conflicts, and nesting removal in DPLL driver function

In [10]:
import json
from pathlib import Path
from argparse import ArgumentParser
from dimacs_parser import DimacsParser
from model_timer import Timer

# import numpy as np

# input_file = '../input/C459_4675.cnf' 
# input_file = '../input/C1597_081.cnf'
input_file = '../input/C1065_064.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
print(instance, end="")

Number of variables: 50
Number of clauses: 1065
Variables: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50}
Clause 0: {35, 6, -24, 22, -1}
Clause 1: {39, 41, -48, 29, 30}
Clause 2: {-32, -31, 16, -4, -1}
Clause 3: {37, -27, -17, -42, 29}
Clause 4: {-29, 4, -15, -47, 22}
Clause 5: {35, 36, 37, 6, -49}
Clause 6: {3, 45, -46, -35, 31}
Clause 7: {-19, 49, -44, -39, -2}
Clause 8: {-27, 39, 42, -36, -1}
Clause 9: {-30, -50, -16, 27, -4}
Clause 10: {-25, 7, -18, -40, -36}
Clause 11: {10, -18, 15, 19, -39}
Clause 12: {11, -19, -47, -42, 31}
Clause 13: {-18, -43, 24, -39, -38}
Clause 14: {-26, -50, 14, -48, 24}
Clause 15: {-29, 12, -46, -5, -2}
Clause 16: {34, 8, -16, -6, 27}
Clause 17: {-30, 40, 10, 44, 28}
Clause 18: {40, -23, -48, 19, 24}
Clause 19: {-31, 35, -19, -50, 48}
Clause 20: {36, -25, -22, 44, -16}
Clause 21: {-29, 38, 46, -39, -5}
Clause 22:

In [11]:
symbols = list(instance.vars)
clauses = [list(clause) for clause in instance.clauses]
model = {}

In [12]:
def eval_clause(clause, model):
    unassigned = False 

    for var in clause:
        if (abs(var) in model):
            value = model[abs(var)]

            if (var > 0 and value) or (var < 0 and not value):
                return 'TRUE' 
        else: 
            unassigned = True 
    if unassigned:
        return 'UNKNOWN' 
    
    return 'FALSE'

In [13]:
def eval_instance(clauses, model): 
    every = True 
    for clause in clauses: 
        clause_value = eval_clause(clause, model) 

        if clause_value == 'FALSE': 
            return 'UNSAT' 
        if clause_value != 'TRUE': 
            every = False 
    if every:
        return 'SAT' 
    
    return 'UNKNOWN'

In [14]:
def pure_symbol(clauses, model):
    pure = {} 
    impure = set()
    for clause in clauses: 
        if eval_clause(clause, model) == 'TRUE': 
            continue 

        for x in clause: 
            var = abs(x)

            if var in model or var in impure:
                continue

            if var not in pure:
                pure[var] = x > 0

            else: 
                if pure[var] != (x > 0): 
                    pure.pop(var)
                    impure.add(var)
    if not pure: 
        return [], []
    return list(pure.keys()), list(pure.values())

In [15]:
# unit clause 
def unit_clause(clauses, model): 
    model = model.copy() 
    new_assignments ={}
    found_unit_clause = True 

    while found_unit_clause: 
        found_unit_clause = False 

        for clause in clauses:
            clause_val = eval_clause(clause, model)
            if clause_val == 'TRUE':
                continue 
            elif clause_val == 'FALSE':
                return None, None
            
            unassigned = [lit for lit in clause if abs(lit) not in model]

            if len(unassigned) == 1: 
                literal = unassigned[0]
                if (abs(literal)) in model: 
                    continue 
                else: 
                    model[abs(literal)] = literal > 0
                    new_assignments[abs(literal)] = literal > 0
                    found_unit_clause = True

    return list(new_assignments.keys()), list(new_assignments.values())

Pick most frequently occuring variables in unsatisfied clauses

In [16]:
def pick_best_branching_var(symbols, clauses, model):
    var_score = {var: 0 for var in symbols}

    for clause in clauses:
        if eval_clause(clause, model) != 'TRUE':
            for lit in clause:
                var = abs(lit)
                if var in symbols:
                    var_score[var] += 1

    if not var_score:
        return None
    return max(var_score, key=var_score.get)

In [18]:
def dpll(clauses, symbols, model): 
    
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        return model 
    elif instance_status =='UNSAT':
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        return dpll(clauses, symbols, model)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        return dpll(clauses, symbols, model)
    
    # branch
    if not symbols:
        return None

    p = pick_best_branching_var(symbols, clauses, model)
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    res = dpll(clauses, rest, (model | {p: True}))
    if res is not None:
        return res
    
    return dpll(clauses, rest, (model | {p: False})) # backtracking with False

In [19]:
result = dpll(clauses, symbols, model)

if result is not None:
    print(f'Result: SAT, Solution: {result}')
else:
    print('UNSAT')

UNSAT
